# MAMMQA hop-trace run: 300 MMQA dev questions, released prompts

This notebook produces full per-agent traces of **MAMMQA as released** on 300 MultiModalQA dev questions,
so that `hop_audit` can test the pre-registered claim: *most wrong answers happen after the bridge
information is already inside the pipeline*. See `PREREGISTRATION.md` in the project's `hop_audit` folder.

| | |
|---|---|
| **Questions** | 300 dev questions with at least one latent gold hop. Frozen list (sha256 `680d34ad38ea7d7a…`), stratified by type, shuffled order. |
| **Pipeline** | `prompts.py` verbatim (sha256 checked), `agents.py` message formats, 7 agents. The paper's per-question context (`text_doc_ids`, `table_id`, `image_doc_ids`) with official pixels and **no image titles**. |
| **Only deviation** | The aggregator ends with `<answer>…</answer>`, so EM is defined. |
| **Model** | Qwen2.5-VL-7B-Instruct, 4-bit NF4, greedy. **One full copy per T4**, two workers in parallel. |
| **Leakage** | Prompts are built from gold-free items. Predictions are frozen and hashed before gold is read. |

**Settings:** accelerator **GPU T4 x2**, Internet **on**, inputs `adibraian/multimodalqa-dataset` and
`final-dataset-images-mmqa`. Use **Save & Run All** (a background commit, up to 12 h).

**Resume:** if the run stops before 300, add this notebook's own output as an input and run it again.
Finished questions are skipped.

In [ ]:
# Cell 1 — environment
import os, sys, subprocess, importlib, json, re, time, math, random, hashlib, gzip, zipfile, string, shutil, glob
from pathlib import Path
from collections import Counter, defaultdict
NOTEBOOK_START = time.time()
MOCK = os.environ.get("TRACE_MOCK", "0") == "1"          # local plumbing test only; never set on Kaggle
if not MOCK:
    if importlib.util.find_spec("bitsandbytes") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.45"], check=False)
    import torch
    N_GPU = torch.cuda.device_count()
    print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
    import transformers; print("transformers", transformers.__version__)
else:
    N_GPU = int(os.environ.get("TRACE_MOCK_GPUS", "2"))
print("MOCK =", MOCK, "| workers =", N_GPU)

In [ ]:
# Cell 2 — configuration (frozen; edit only MAX_RUNTIME_HOURS or MODEL_PATH_OVERRIDE)
INPUT_ROOT = Path(os.environ.get("TRACE_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(os.environ.get("TRACE_OUTPUT_ROOT", "/kaggle/working/mammqa_trace"))
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_PATH_OVERRIDE = None
MAX_RUNTIME_HOURS = float(os.environ.get("TRACE_MAX_HOURS", "11.0"))   # stop STARTING questions after this
SELECTION = {"seed": 20261030, "n": 300, "criterion": "MMQA dev, >=1 latent gold hop (hop_audit.gold), excluding 52 previously inspected qids", "pool_sizes": {"Compare": 173, "Compose": 676, "Intersect": 91, "TableQ": 117}, "allocation": {"Compare": 49, "Compose": 192, "Intersect": 26, "TableQ": 33}, "questions": [{"qid": "f1d8bf21d404686f93b0993778905686", "top_type": "Compare"}, {"qid": "b18df92399d5d28a9e28236f3046c50f", "top_type": "Intersect"}, {"qid": "e7e728212dd999b3cf1a8de67442d3ac", "top_type": "Compose"}, {"qid": "983896c1506e70dabb50e77a717268d9", "top_type": "Compose"}, {"qid": "87d0d1beddda61d9333506028006c7bd", "top_type": "Compose"}, {"qid": "b04f9ebb5c841438872ffd1342c788e5", "top_type": "Compare"}, {"qid": "96aad9d0dc7fcb9fdea8db970f90853a", "top_type": "Compose"}, {"qid": "062686b4df599f4111761a0753f4c6c1", "top_type": "TableQ"}, {"qid": "4ea83a52d230178274421f426d8e3dc3", "top_type": "TableQ"}, {"qid": "cc5273a106f1c124aab46feb4f9461b0", "top_type": "Compose"}, {"qid": "1d96824848914bb5ab82bd87085605ce", "top_type": "Compare"}, {"qid": "12b5572cf9bbb103a607a9e0b880c94a", "top_type": "Compose"}, {"qid": "8fba1443e531d82871ccbd5a8ab9f3be", "top_type": "Compare"}, {"qid": "380f72986c6728b13c17f77dc6c22205", "top_type": "Compose"}, {"qid": "13c2a1185b4afbdb3ff93e673e6aa82a", "top_type": "Compose"}, {"qid": "591fea517306d7aaa6d40ab6d92c60c7", "top_type": "TableQ"}, {"qid": "74f9019633803db0ac332ee70ff326b0", "top_type": "Compose"}, {"qid": "ff7ecb97b47ba179fb7b7ee98d488d1a", "top_type": "Compose"}, {"qid": "3e0e050e933e43ccb163b7fcc7535b21", "top_type": "Compose"}, {"qid": "6cdbdb112202d243e6a061fab81e9b7a", "top_type": "Compose"}, {"qid": "c900076a1a7a0d6a586e6e5dc7d60696", "top_type": "Compose"}, {"qid": "24d401e9e9842ac66ee811ea21fbb509", "top_type": "Compose"}, {"qid": "c38d39b39912cc58a0822432b6660fbf", "top_type": "Compose"}, {"qid": "ede9a615811ae817b7e7e4cce658b595", "top_type": "Compose"}, {"qid": "c49e7cfc4919fe9ca4a361579c582e39", "top_type": "Compose"}, {"qid": "ddc3ef7735c55abceb3dcd2c0b863498", "top_type": "Compose"}, {"qid": "20f925030747b53b816d9f5436a6109c", "top_type": "TableQ"}, {"qid": "ab3c347c802bfdfd80b53320400e263c", "top_type": "Compose"}, {"qid": "3381825f509615f7d458fd6205d07e77", "top_type": "Compose"}, {"qid": "042167257f2a553469fa2c3bfef659f6", "top_type": "Compose"}, {"qid": "0a8b8b970675ee05c9579f4330c19d70", "top_type": "Intersect"}, {"qid": "50e892833f89d565b6f5dc57a237b325", "top_type": "Compose"}, {"qid": "e1121329fd8a39084973981b0ae09f4a", "top_type": "Compose"}, {"qid": "b8b26f0400eb24a66b16445077a0a81e", "top_type": "Compose"}, {"qid": "e4fedf45de5b3fdb19ab04f43bb2377a", "top_type": "Compose"}, {"qid": "57e8c0c3de0b05088ff7a3c85edef187", "top_type": "Compose"}, {"qid": "69389b694bb9a06f7f023eadf8ba1f61", "top_type": "Compose"}, {"qid": "10af21e2b2756b42cc049dbbd73fe4ef", "top_type": "Compose"}, {"qid": "daf24c25a5126e8caf6773a7cd956298", "top_type": "Compose"}, {"qid": "e368b925129bc3ad35a94a6d5e9de770", "top_type": "Compose"}, {"qid": "b34e6f73a8e2041acfdaa10be9d590bc", "top_type": "Compose"}, {"qid": "12cc070c25eedb638ec9b018e4678156", "top_type": "Intersect"}, {"qid": "f45e8147c2d7cd59000bb36574120900", "top_type": "Compare"}, {"qid": "3d1cfc992ee4774dba9a663ca9b36809", "top_type": "Compare"}, {"qid": "6ff171b80a0f0f8144a65f16be37b24c", "top_type": "TableQ"}, {"qid": "7bd7f567ed3392942c746c652e356be8", "top_type": "Compose"}, {"qid": "1ad0c10aa88ae1ee3af333e942b146a8", "top_type": "Compose"}, {"qid": "e723bdfa758e95165ffdbb99dab1bdc2", "top_type": "Compose"}, {"qid": "ff9aba2705aed6a76fe35afa2ad24d6f", "top_type": "Compose"}, {"qid": "8f175a0cb4fa6ff5d62cb680638c98eb", "top_type": "Compose"}, {"qid": "86d78c3d0b404a680cdb247e15babd64", "top_type": "TableQ"}, {"qid": "445bc5e02f240a4d151b30e9ec6dea80", "top_type": "Compose"}, {"qid": "49ee63cd9334f492856f4f03aff1807e", "top_type": "Compose"}, {"qid": "650e85cfb29a3e913d55831c30090080", "top_type": "Compare"}, {"qid": "52d8d419290b16a96633cd9920f1ea8c", "top_type": "Compose"}, {"qid": "ac9c96e3b92ac9227a8a5f2ccea4d780", "top_type": "Intersect"}, {"qid": "2d4201adb565f16f8024fe3225f301dd", "top_type": "Compose"}, {"qid": "28411052506c4009e3707aefe55fab15", "top_type": "Compose"}, {"qid": "9836e53dcac8847a9af8e414b2d68cf2", "top_type": "Compose"}, {"qid": "3c464ec9ff6ae28426640fdd51aaf2fd", "top_type": "Compose"}, {"qid": "6780758fb1ebc547582c186ca681b84d", "top_type": "Compose"}, {"qid": "c0674458cd26df97b8e5579d871016e5", "top_type": "Compose"}, {"qid": "09806284de195b3747072c1b4df7b034", "top_type": "Compose"}, {"qid": "67e67f6e04a508420dbf6e0736038686", "top_type": "Compare"}, {"qid": "375ab64096ab9e840266206aff6f461a", "top_type": "Compose"}, {"qid": "bf39d84c7e1ce3e0f7b98c1b8e4be6ef", "top_type": "Intersect"}, {"qid": "05cf9d2b18e4c0358e321dbd28b12d14", "top_type": "Intersect"}, {"qid": "e9cfe3d149fccbf737f3cca2ab607e46", "top_type": "Compare"}, {"qid": "c94e7a70bb7f9b7ee82c7a4320eb756a", "top_type": "Compose"}, {"qid": "ac72e14d54369a683fa34bf74da17395", "top_type": "TableQ"}, {"qid": "71c65b4e835609c89fff04ef226eb27f", "top_type": "Compose"}, {"qid": "fdd0c88d5c107669cc4b6813894b7f5e", "top_type": "Compose"}, {"qid": "6b0c1ee8eaaa43d13e176b4833a0ecb3", "top_type": "Compose"}, {"qid": "a2f15e58d703567c1a32d440ddf30299", "top_type": "Compose"}, {"qid": "c932bd00be5ba7df47e96f7f18c69eda", "top_type": "Compose"}, {"qid": "1982ea14d6fa923927619d4a4c68db87", "top_type": "Compose"}, {"qid": "2905288d5e473cf4a199a909aac1be93", "top_type": "Compare"}, {"qid": "378f2ab361968ed2720f48eface3889a", "top_type": "Compose"}, {"qid": "e9d115431bb3af91b67f8d57b3c8616a", "top_type": "Intersect"}, {"qid": "e3adf9992eeb8686e5001ac112de7178", "top_type": "Compose"}, {"qid": "39bd3f9ddc4b33aee21bce4774d54834", "top_type": "TableQ"}, {"qid": "871e09ce0d6dd22c33e2abc5e070ded3", "top_type": "Compose"}, {"qid": "9475edd08caab8f60fc5f4e14be760b0", "top_type": "Compose"}, {"qid": "16ad0ff40da24c7330ca490987dadf87", "top_type": "Compose"}, {"qid": "6273c933272b9332f9a70bf27dd84e68", "top_type": "Compare"}, {"qid": "265c9d2214f57787e65302039ec8b8f9", "top_type": "Compare"}, {"qid": "bc2e226b953fe9073c646c40b6fbbe73", "top_type": "Compose"}, {"qid": "7304d0bba0ec848c223e4b203e2d0158", "top_type": "Compose"}, {"qid": "a9c9a80d9b266cf786ebd2b3cba0d60f", "top_type": "Compose"}, {"qid": "c24d916959a91e823faa7a8ce8292bc7", "top_type": "Intersect"}, {"qid": "d4f7ddfba90360ad1e0fb4c100b7ab41", "top_type": "Intersect"}, {"qid": "82edc4d2853f5563dbce0001a3944d20", "top_type": "Compose"}, {"qid": "51f16e3b8adb750b07683daf4487094d", "top_type": "Compose"}, {"qid": "394605f7d9fd8c2bd078361bc26ada7c", "top_type": "TableQ"}, {"qid": "63892c41545d011066d4a5e9a75dc71d", "top_type": "Compose"}, {"qid": "c07c108b8be892dd32e4485406ba61b4", "top_type": "Intersect"}, {"qid": "f11cff3920ce9f8a27dc4fe961f10d5c", "top_type": "Compare"}, {"qid": "ebbc4d5836afde56f6c50d6f06deff2e", "top_type": "Compose"}, {"qid": "c328bffe44419f1ef817569e82fc6d2c", "top_type": "Compare"}, {"qid": "7ad698710c7ccb4f92bde540a1f189e4", "top_type": "Compose"}, {"qid": "d8a3cc4403fbcb2ea99382815730ddec", "top_type": "Compose"}, {"qid": "f378e6c2b3178511b6987480dc57f417", "top_type": "Compose"}, {"qid": "fe687c0c73f3094669d5ec9eaddc0e15", "top_type": "Compare"}, {"qid": "111e2c6d6aac95d1ebd632d2e37a70e7", "top_type": "Compose"}, {"qid": "c93e3959754568b3799ea4bd6b355818", "top_type": "Compose"}, {"qid": "2cca87e7b1b77ee2e3f4f83119c1854c", "top_type": "Intersect"}, {"qid": "0efd517e21041f77d5bd4827eb93391e", "top_type": "Compare"}, {"qid": "f32edadabeda35d4aaf685ca937386a9", "top_type": "Compose"}, {"qid": "069ac34398c0b27d57efddea30e510e4", "top_type": "Intersect"}, {"qid": "d27caed0b360cd98c8b04c53751582c8", "top_type": "Compose"}, {"qid": "31979c4f4273dfe571df8d50d268bf8b", "top_type": "TableQ"}, {"qid": "966c44b2a49b38286ff313a2386824cc", "top_type": "Compose"}, {"qid": "a1e42da526ff18d90cd2eda775fc3be2", "top_type": "Compose"}, {"qid": "8ee6828df3f7dff3e979d0b87a2e812e", "top_type": "Compose"}, {"qid": "23784e8b16f3c60960174e8b7e94d117", "top_type": "Compose"}, {"qid": "ee7f5fc0979ce89e85a801542edd8ad4", "top_type": "Compose"}, {"qid": "1f4c0c5d7a5fe2d4bac22baf5bef68c2", "top_type": "Intersect"}, {"qid": "5f0702dcdf0137e481d95b62ae6ea23d", "top_type": "Compose"}, {"qid": "76d853b0293fed49f6faa47b17057a6b", "top_type": "TableQ"}, {"qid": "29af2f314cb813aead8134609ecca5b1", "top_type": "Compose"}, {"qid": "c03a4eff02cfcf14693ab23434683fb9", "top_type": "Compose"}, {"qid": "b7106bcf7704805ca0ba84ddd85004f4", "top_type": "Compare"}, {"qid": "bed46f5100d7cab5c4f1115a7c3638be", "top_type": "TableQ"}, {"qid": "c81c4000f886ed9ebe29f8989088575e", "top_type": "Intersect"}, {"qid": "62bcc955094e4f2dc3e8a3b41384b58d", "top_type": "Compose"}, {"qid": "796d86b407796b748d9feb9e9187a898", "top_type": "TableQ"}, {"qid": "cad2a8c3d084b5c5f32b28954a00702b", "top_type": "Compose"}, {"qid": "a4a3b369ef71a1cb710e993ac313812c", "top_type": "Compose"}, {"qid": "3590ab0420e71d37892fcfda94aee409", "top_type": "TableQ"}, {"qid": "d771de45b0645ba154948c0db000d3be", "top_type": "Compare"}, {"qid": "189b3d7c3139f85469bfc72e423a9b44", "top_type": "Compose"}, {"qid": "424e731dec99fd3b36fb5ba2c59d1f16", "top_type": "Compose"}, {"qid": "540ceea7e372fedb11d29561059d974e", "top_type": "Compose"}, {"qid": "41a0f1050d12b5ef209a19dd4a4822b3", "top_type": "Compose"}, {"qid": "7d88a7aeac7eb0f69c74c1f642ed48c0", "top_type": "Compose"}, {"qid": "0f22b5d312a0b8cc97455e66ddda260c", "top_type": "Compose"}, {"qid": "e1fbbed63ee293abb898a5de10fefc69", "top_type": "TableQ"}, {"qid": "7adcb26fda18602af51a8b9f2061f94f", "top_type": "Compose"}, {"qid": "ab92d78f544f09ec7b61496ece7c0833", "top_type": "Compose"}, {"qid": "74e41eba24bcaeb1212cb23425f1e69e", "top_type": "TableQ"}, {"qid": "4778c9f3b15c46c0f7cb8779c3ca4b1c", "top_type": "Compare"}, {"qid": "a5e00e7fd6d976c2b4b397c47ab40014", "top_type": "Intersect"}, {"qid": "c528ad759ef7277b55426514b219bffb", "top_type": "Compose"}, {"qid": "df6a224711b6900ef8e1a91e057b4865", "top_type": "Compose"}, {"qid": "4a870e608f53021d46b2ee4bd3c9eea8", "top_type": "Compare"}, {"qid": "0f0b4253d4f4b8e75740308b6a73619b", "top_type": "Compose"}, {"qid": "6bce25aa30561e06dd9cb3bb1003b627", "top_type": "Compose"}, {"qid": "9264216e2fb4bb4868a823e887d09783", "top_type": "Compose"}, {"qid": "a0c18732582c6572f58774c7ae4543fd", "top_type": "Compare"}, {"qid": "d8a47742253fc5049ddeeef9f495f7b2", "top_type": "Compare"}, {"qid": "11f0f941600ad6c30e9af93cc9f51712", "top_type": "Compose"}, {"qid": "d7c27ea21b8462542f1c66c526418200", "top_type": "Compose"}, {"qid": "5a92dd84de9fc1a8fa23cc73483d2753", "top_type": "Compose"}, {"qid": "29ff34fcffe7fc9ebeb42bb1b1fd826b", "top_type": "Compose"}, {"qid": "65dfcfcd8831e3740f9ad1f9728c6adf", "top_type": "Compose"}, {"qid": "950e8463e15d9ae0dadd4d25bc4e9b5d", "top_type": "Compose"}, {"qid": "781e728f3710587c2eb30edee58f2021", "top_type": "Compose"}, {"qid": "e8874819c8c171d83e26e0ae0f4d2ce4", "top_type": "Compose"}, {"qid": "4f95c12e4d2d0b0eb114f3e91db5fbc1", "top_type": "Compare"}, {"qid": "ed06a4b34fa212d6be4522a017d58b82", "top_type": "TableQ"}, {"qid": "07b2f4699287ff38512d66e98b250014", "top_type": "Compare"}, {"qid": "c3abb465eda6c041d19c7cac507593ed", "top_type": "Compose"}, {"qid": "b85822a4402c85f6f77dec45014232a0", "top_type": "Compare"}, {"qid": "04dfd7512cbb416af41988232cdd0c68", "top_type": "Intersect"}, {"qid": "682370aa53895c2ba3cd49037167518b", "top_type": "Compose"}, {"qid": "1a6f5a7e7e93514a7515977a3511a822", "top_type": "Compose"}, {"qid": "7a892a574302f8422626eac74b696b95", "top_type": "Compose"}, {"qid": "64abc23e9b46cbe9da333347dbd60c1e", "top_type": "Compose"}, {"qid": "f44867274c5f1bf9a001f953c86754f1", "top_type": "Compose"}, {"qid": "81088e10057564782ada42345f1b51bc", "top_type": "Compose"}, {"qid": "287040504a31a4770a12ccb07c36eb13", "top_type": "Compose"}, {"qid": "014f0f79f9af827ba68cbf1b65130f51", "top_type": "Compose"}, {"qid": "7e2b401896aeccf738c7397de9e8d7cf", "top_type": "Compose"}, {"qid": "b11b7bb67eb7917ca109edb62828f618", "top_type": "Compose"}, {"qid": "e6bf1ac6cc85c5563ae4a2dc1ce33d5f", "top_type": "TableQ"}, {"qid": "04bbb16534dfab370b011a84d70bf051", "top_type": "Compose"}, {"qid": "e8eec540759053e716f0d4a1010e199f", "top_type": "Compose"}, {"qid": "5fa187a053d883f7d8f70660ecb8dcdb", "top_type": "TableQ"}, {"qid": "7c1c225becb07847f5f84e6af0417b7b", "top_type": "Compare"}, {"qid": "806102fe5327afe00614677954db5579", "top_type": "Compose"}, {"qid": "e67df619b8e0d9bf4eb1d7740505fd88", "top_type": "Compose"}, {"qid": "a8c6be32a981a89d6f99e81de18f0b2a", "top_type": "Compose"}, {"qid": "082e679ef7335d449e250974bacc6101", "top_type": "TableQ"}, {"qid": "7faa3a358a3e25fbf959d54fa05a6e91", "top_type": "Compare"}, {"qid": "3be96f38162933c89eda66879560e41e", "top_type": "Compose"}, {"qid": "15e4aa743bf354ea03704de66e12b782", "top_type": "Intersect"}, {"qid": "a2683d2d5cb57888f39f06ed08115ba1", "top_type": "Compose"}, {"qid": "2056dcf5ed1c70cb799c09a023a4f73c", "top_type": "Compose"}, {"qid": "645a861de74604e8025caf759f425748", "top_type": "TableQ"}, {"qid": "3e99b830ec0f40e6ad9b94766200bb9e", "top_type": "Compose"}, {"qid": "168d9f74f7f355cc97139f9a2814b69d", "top_type": "Compose"}, {"qid": "c8c72cfb2d1f4e8fbfb77f876f8262f6", "top_type": "Compose"}, {"qid": "854f0e26aba9770919a08d5ed6a54cac", "top_type": "Compose"}, {"qid": "9c3996fcf2af4de7daa540e1e2286470", "top_type": "Compose"}, {"qid": "17bea087783c5c4a2d4ee6ffb21920e3", "top_type": "Compose"}, {"qid": "54e4a753600ab1ecc963dd11a5726116", "top_type": "Intersect"}, {"qid": "cc2d57102e3fc11d7aabc7b1f3b11e1f", "top_type": "Compare"}, {"qid": "9d249d4e530d7a3cdc9807f1e7cd584f", "top_type": "Compose"}, {"qid": "3f14c9a8448f16f5d983667eee0bf65e", "top_type": "Compose"}, {"qid": "ab47a6769bedf581f22c8db06a36361f", "top_type": "TableQ"}, {"qid": "725ca323b783e2b010c73a88a938a370", "top_type": "Compose"}, {"qid": "220f55e31592a5ed8e804402e012a39f", "top_type": "Compose"}, {"qid": "97dfcada221ed75c850c5a1dc9965a0d", "top_type": "Compare"}, {"qid": "f356667690a995336f76344d097bb595", "top_type": "Compare"}, {"qid": "e6ebef376a5f8d390513c0942a3861ef", "top_type": "Compose"}, {"qid": "1596e3f73a33e24da59a19a2dece6cbb", "top_type": "Compose"}, {"qid": "9ced6c0c15cabb527367db9dce03e59a", "top_type": "Compose"}, {"qid": "2f34ec15d30206535baa99d709903307", "top_type": "TableQ"}, {"qid": "e6820a7887a5ed27294ae43fad3445f9", "top_type": "Compose"}, {"qid": "f2c60a572ce83bacbf78cda7dc625b0c", "top_type": "Compare"}, {"qid": "e562298d120bab0803434d2784357755", "top_type": "Compose"}, {"qid": "4437c68c9183d1abbf460c40769dc071", "top_type": "Compose"}, {"qid": "8824955c9280e6907177511dda24d7e4", "top_type": "TableQ"}, {"qid": "911f0e3da26d46f30316abb7d56fd163", "top_type": "TableQ"}, {"qid": "d275b09a6529489133dae0d8ce678043", "top_type": "Compose"}, {"qid": "db0e7104ed87fbb5abe784501c9c86a1", "top_type": "Intersect"}, {"qid": "daa6355a6b176227813640a8b9be4353", "top_type": "Compose"}, {"qid": "5980eb0068c2d2a2ec2193ff2f8696e8", "top_type": "Compare"}, {"qid": "c132fc928153e2009b49449383b28f9e", "top_type": "Compose"}, {"qid": "ce4c783ec908dc4139773ba5e6ed6d88", "top_type": "Compose"}, {"qid": "03b001f3f7c21e7ef83f05df77e66d4f", "top_type": "Compare"}, {"qid": "493ce9b7696dc9a3c048cbbe8930031c", "top_type": "Compose"}, {"qid": "f0a47f05569b501151bc6aac89a2c89d", "top_type": "Intersect"}, {"qid": "723cff08515cf6f04223f8218c481b0c", "top_type": "Compose"}, {"qid": "57d2f3b4f7f8ebbb7c8f35d730fcceb5", "top_type": "TableQ"}, {"qid": "1250357fe7efc2308c84e0c0b580e2c9", "top_type": "Compose"}, {"qid": "9e1d388983bdccc4f4f3cce86c9cfbc7", "top_type": "Compose"}, {"qid": "043d8199746ded4fba667a607684173a", "top_type": "Compose"}, {"qid": "84a89298751f5034887bb17df265bcfd", "top_type": "Compose"}, {"qid": "356070d6932cb55157b6163b819c8f21", "top_type": "Compose"}, {"qid": "d96a20235dd15e1eb8544ad9d28494e2", "top_type": "Compose"}, {"qid": "9062f0a10733ef0abf180e6788feca29", "top_type": "TableQ"}, {"qid": "aa50d519067acf344d4b01187575b13e", "top_type": "Intersect"}, {"qid": "677018adc8228b876dbd2967b05634ec", "top_type": "Compare"}, {"qid": "71aa5ca0adee50856ac97ff2280b4dfc", "top_type": "Compose"}, {"qid": "651a585e5b5500161eb187e87677a16c", "top_type": "TableQ"}, {"qid": "3c76a1f8ad00a1c6928a65a370f48cbf", "top_type": "Compose"}, {"qid": "57ed1695ef2b175552dbcf597f975d71", "top_type": "Intersect"}, {"qid": "3e01211c7699437a3432146b9e374d55", "top_type": "Compose"}, {"qid": "0318e3b1eb51736bf111d233a396fdbf", "top_type": "Compose"}, {"qid": "7873682e71d6f76b235c48fbfbde72b7", "top_type": "TableQ"}, {"qid": "3a7faddd273f1364ac9fbb66b6658fe9", "top_type": "Compare"}, {"qid": "cc9541815a25b26ca612f9895c273925", "top_type": "TableQ"}, {"qid": "8b8caa13c597e3c0b5656eb1ff1e816f", "top_type": "Compose"}, {"qid": "f84b94c833d89d1bd23d2831fd45229c", "top_type": "Compare"}, {"qid": "886f92bcdd89eb6b7ac0f2e069d3b730", "top_type": "Compose"}, {"qid": "830197a234920e4ab599a35b47974ea0", "top_type": "Compare"}, {"qid": "b0a785f3f132bf18d9fcf1d1f8dea298", "top_type": "Compose"}, {"qid": "52fa3798ab4a7626072f3e47b329789e", "top_type": "Compose"}, {"qid": "8216c175a7d406fedd3d163d8ed6c387", "top_type": "Intersect"}, {"qid": "0e9d96fb7b5f17969db33f2e613b20ee", "top_type": "Compose"}, {"qid": "216ff289c079b301de87c96270d371ba", "top_type": "Compare"}, {"qid": "9418e274d0aa06e5cefc3ab80219351a", "top_type": "Compose"}, {"qid": "de22c413b35b616bf5123f20e8c97449", "top_type": "Compose"}, {"qid": "8327f9a26ec1ce848a80e2500f70929a", "top_type": "Compose"}, {"qid": "1f3664fd1742fa7c5b54fd39300c6534", "top_type": "Compose"}, {"qid": "42e8a744df1cab9aa5ad0964fa216813", "top_type": "Compose"}, {"qid": "6fac63e14f667e265c0451b184566a47", "top_type": "Compare"}, {"qid": "dd920fa1a56fece13a4585ff36c7e016", "top_type": "Compare"}, {"qid": "40b3e2453fe619126174d64bec19157c", "top_type": "Compare"}, {"qid": "7cb75d90b1499793f280d0ec828cc008", "top_type": "Compare"}, {"qid": "2796902e5aa2f7bcfaf8f91cd032aea2", "top_type": "Compose"}, {"qid": "37675a30a1c36e162a68f1e0ff8be63f", "top_type": "TableQ"}, {"qid": "14cfca87a84d25ca17313ad43e0b8f88", "top_type": "Compose"}, {"qid": "3b8c10bbf59a3d8722dcbb3a3cfd3575", "top_type": "Compare"}, {"qid": "119c9a9bf38ca9f84a3900d7c7fd1fe5", "top_type": "TableQ"}, {"qid": "974c10e442226d4f9e1e042aca404a0c", "top_type": "Compare"}, {"qid": "691dfdcfe1a2884998f1fe60b6f7ef15", "top_type": "Compose"}, {"qid": "a7056d9fc4bb2bffec3e7f3a9a72a61a", "top_type": "Intersect"}, {"qid": "d2322ed754451eb703f6c1379f0535e8", "top_type": "Compose"}, {"qid": "81712f380ca5dd8d58c333fc1c366ef3", "top_type": "Intersect"}, {"qid": "1946869d518678a95a5d2d51961818aa", "top_type": "Compose"}, {"qid": "35ffa88a447dd6ca8ed2d90564ed87a8", "top_type": "Compose"}, {"qid": "a1b40c12a33c4e429cd72d8a88899454", "top_type": "Compose"}, {"qid": "87f95c06500b8852cac891a2697cc2f1", "top_type": "Compare"}, {"qid": "4e09696e32e1a916372d090e986af012", "top_type": "Compose"}, {"qid": "022a4190652d4999aa326d64a6bf87a4", "top_type": "TableQ"}, {"qid": "9c95db901f2796d3207c152fdd9391ad", "top_type": "Compose"}, {"qid": "ee790547f5facabb581c8806e4babdeb", "top_type": "Compose"}, {"qid": "0784b43a6c6eeddb318f65ea2baf397e", "top_type": "Compose"}, {"qid": "4cdb083b7d32719ff0ed63588765de4d", "top_type": "Compose"}, {"qid": "7991b4f0ed0106d4693f18194e767658", "top_type": "Compare"}, {"qid": "7a8a030df747a67a6715393e82e7c723", "top_type": "Compose"}, {"qid": "20eeccd19adb6f739680f234c37314ed", "top_type": "Compose"}, {"qid": "32fa293b30513e3619128925a64389a1", "top_type": "Compare"}, {"qid": "e1238a0f20b1360ddbf16448f57484d3", "top_type": "Compose"}, {"qid": "f6e543bad0cfd5933910a923705275ea", "top_type": "Compose"}, {"qid": "fcf2b36770bef483cbc122713f17f125", "top_type": "Intersect"}, {"qid": "af3e03d30cebb1708b672ab3f238b827", "top_type": "Compose"}, {"qid": "782bfbc63ccd5db12e1efe9afd9e0752", "top_type": "Compose"}, {"qid": "e3027465630e99e8f964a29664034467", "top_type": "Compose"}, {"qid": "7b5a18fe0ff6b814cc3e7e79d58dd314", "top_type": "Compose"}, {"qid": "550f9f620d53ee8077154847af3b4651", "top_type": "Compare"}, {"qid": "beb5d85dc0811603152bb64c768dd412", "top_type": "Compose"}, {"qid": "78eab38140bfb9b730f155eb83e18ca8", "top_type": "Compose"}, {"qid": "bb4dfe5176aa20b3b24f42c864c6ebb6", "top_type": "Compose"}, {"qid": "1f23c6976ee393391a9cbf26421ed1ac", "top_type": "Compare"}, {"qid": "16f832a8ae461b35dff86a41827b3179", "top_type": "Compose"}, {"qid": "8fe79ee97c1ac904fc538e861d9b7c9e", "top_type": "Compose"}, {"qid": "77f3a5b3c0707ded5197accd871bfe17", "top_type": "Compose"}], "sha256_questions": "680d34ad38ea7d7a05a5c82fd701b8fe3fe2d3fa0e9dc2ed74d47ce82918e7e4"}
assert hashlib.sha256(json.dumps(SELECTION["questions"], sort_keys=True).encode()).hexdigest() == SELECTION["sha256_questions"]
ORDER = [q["qid"] for q in SELECTION["questions"]]
N_LIMIT = int(os.environ.get("TRACE_N", str(len(ORDER))))
ORDER = ORDER[:N_LIMIT]
OFFICIAL_PROMPT_FINGERPRINTS = {"stage1": "8446a999f82eddf7", "stage2": "74b0378447dc51b4", "stage3": "a56057c4b5558510"}
DIRS = {k: OUTPUT_ROOT / k for k in ["config", "inputs", "cache", "checkpoints", "errors", "logs", "predictions", "metrics", "images"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(DIRS["logs"] / "main.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")

def atomic_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp"); tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8"); tmp.replace(path)

log(f"{len(ORDER)} questions | selection sha256 {SELECTION['sha256_questions'][:16]} | allocation {SELECTION['allocation']}")

In [ ]:
# Cell 3 — locate MMQA files and official image pixels
def walk_files(root):
    for dp, _, fns in os.walk(root, followlinks=True):
        for n in fns:
            yield Path(dp) / n

def read_jsonl(path):
    opener = gzip.open if path.name.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

ALL_INPUT_FILES = [p for p in walk_files(INPUT_ROOT)]
def find_mmqa(kind):
    hits = sorted([p for p in ALL_INPUT_FILES if f"MMQA_{kind}" in p.name and (p.name.endswith(".jsonl") or p.name.endswith(".jsonl.gz"))],
                  key=lambda p: (p.name.endswith(".gz"), len(str(p))))
    if not hits:
        raise FileNotFoundError(f"MMQA_{kind} not found under {INPUT_ROOT}: attach adibraian/multimodalqa-dataset")
    return hits[0]
MMQA = {k: find_mmqa(k) for k in ["dev", "texts", "tables", "images"]}
for k, v in MMQA.items():
    log(f"{k:<7} {v}")

In [ ]:
# Cell 4 — build GOLD-FREE inference items from the paper's haystack, then drop the gold-bearing rows
from dataclasses import dataclass
dev_rows = {r["qid"]: r for r in read_jsonl(MMQA["dev"]) if r["qid"] in set(ORDER)}
assert len(dev_rows) == len(ORDER), "selected qid missing from dev"
need_text = {t for r in dev_rows.values() for t in r["metadata"].get("text_doc_ids", [])}
need_table = {r["metadata"].get("table_id") for r in dev_rows.values()} - {None}
need_image = {i for r in dev_rows.values() for i in r["metadata"].get("image_doc_ids", [])}
TEXTS = {r["id"]: r for r in read_jsonl(MMQA["texts"]) if r["id"] in need_text}
TABLES = {r["id"]: r for r in read_jsonl(MMQA["tables"]) if r["id"] in need_table}
IMAGES_META = {r["id"]: r for r in read_jsonl(MMQA["images"]) if r["id"] in need_image}
log(f"records: texts {len(TEXTS)}/{len(need_text)} tables {len(TABLES)}/{len(need_table)} images {len(IMAGES_META)}/{len(need_image)}")

IMG_EXT = {".jpg", ".jpeg", ".png", ".gif", ".webp", ".bmp", ".tif", ".tiff"}
needed_names = {Path(str(m.get("path", ""))).name for m in IMAGES_META.values() if m.get("path")}
IMAGE_FILES, stems = {}, {Path(n).stem: n for n in needed_names}
for p in ALL_INPUT_FILES:
    if p.suffix.lower() in IMG_EXT:
        if p.name in needed_names and p.name not in IMAGE_FILES:
            IMAGE_FILES[p.name] = p
        elif p.stem in stems and stems[p.stem] not in IMAGE_FILES:
            IMAGE_FILES[stems[p.stem]] = p
missing = needed_names - set(IMAGE_FILES)
for z in [p for p in ALL_INPUT_FILES if p.suffix.lower() == ".zip"] if missing else []:
    try:
        with zipfile.ZipFile(z) as zf:
            for member in zf.namelist():
                n = Path(member).name
                if n in missing:
                    out = DIRS["images"] / n
                    if not out.exists():
                        out.write_bytes(zf.read(member))
                    IMAGE_FILES[n] = out
    except zipfile.BadZipFile:
        continue
    missing = needed_names - set(IMAGE_FILES)
    if not missing:
        break
log(f"official image files resolved: {len(IMAGE_FILES)}/{len(needed_names)}")

def unique_headers(h):
    seen, out = Counter(), []
    for x in h:
        out.append(x if seen[x] == 0 else f"{x}_{seen[x]}"); seen[x] += 1
    return out

def table_markdown(rec):
    t = rec.get("table", {})
    headers = [h.get("column_name", "") for h in t.get("header", [])]
    rows = [[c.get("text", "") for c in row] for row in t.get("table_rows", [])]
    width = max([len(headers)] + [len(r) for r in rows] + [0])
    if not headers or len(headers) != (len(rows[0]) if rows else len(headers)):
        headers = [str(i) for i in range(width)]
    headers = unique_headers(headers)
    esc = lambda s: str(s).replace("|", "\\|").replace("\n", " ").strip()
    lines = ["| " + " | ".join(esc(h) for h in headers) + " |", "|" + "|".join(":---" for _ in headers) + "|"]
    for r in rows:
        r = (r + [""] * len(headers))[:len(headers)]
        lines.append("| " + " | ".join(esc(c) for c in r) + " |")
    return "\n".join(lines)

ITEMS = {}
for qid in ORDER:
    meta = dev_rows[qid]["metadata"]
    texts = [TEXTS[t] for t in meta.get("text_doc_ids", []) if t in TEXTS]
    tab = TABLES.get(meta.get("table_id"))
    paths, miss = [], []
    for iid in meta.get("image_doc_ids", []):
        rec = IMAGES_META.get(iid)
        p = IMAGE_FILES.get(Path(str(rec.get("path", ""))).name) if rec else None
        (paths.append(str(p)) if p else miss.append(iid))
    ITEMS[qid] = {   # the ONLY fields a prompt can see
        "qid": qid, "question": dev_rows[qid]["question"],
        "text_block": "\n".join(f"title: {t.get('title', 'No Title')}, text: {t.get('text', 'No Text')}" for t in texts),
        "table_block": (f"Table title: {tab.get('title', 'No Title')}\n" + table_markdown(tab)) if tab else "No table data",
        "has_table": tab is not None, "image_paths": paths, "missing_images": miss,
    }
FORBIDDEN = {"answers", "metadata", "supporting_context", "intermediate_answers", "type", "modalities"}
assert all(not (set(v) & FORBIDDEN) for v in ITEMS.values())
atomic_json(DIRS["inputs"] / "items.json", ITEMS)
n_img = [len(v["image_paths"]) for v in ITEMS.values()]
log(f"items written | images/question mean {sum(n_img)/len(n_img):.1f} max {max(n_img)} | questions with missing image files: {sum(bool(v['missing_images']) for v in ITEMS.values())}")
del dev_rows                                   # gold-bearing rows are not kept during inference

In [ ]:
# Cell 5 — resume: import checkpoints and call cache from an attached earlier version of this notebook
imported = 0
for ck_dir in [p for p in INPUT_ROOT.rglob("checkpoints") if p.is_dir() and (p.parent / "inputs" / "items.json").exists()]:
    for f in ck_dir.glob("*.json"):
        dst = DIRS["checkpoints"] / f.name
        if f.stem in ITEMS and not dst.exists():
            shutil.copy2(f, dst); imported += 1
    for f in (ck_dir.parent / "cache").glob("*.json"):
        dst = DIRS["cache"] / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
log(f"resume: imported {imported} finished questions from earlier versions")

In [ ]:
# Cell 6 — model source (download ONCE before workers start) and run config
def resolve_model_source():
    if MODEL_PATH_OVERRIDE:
        return str(MODEL_PATH_OVERRIDE)
    for cfg in INPUT_ROOT.rglob("config.json"):
        s = str(cfg.parent).lower()
        if "qwen" in s and "vl" in s and "7b" in s:
            try:
                if json.loads(cfg.read_text()).get("model_type") == "qwen2_5_vl":
                    return str(cfg.parent)
            except Exception:
                pass
    if MOCK:
        return MODEL_ID
    from huggingface_hub import snapshot_download
    return snapshot_download(MODEL_ID, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.py", "*.jinja"])

MODEL_SOURCE = resolve_model_source()
log(f"model source: {MODEL_SOURCE}")
RUN_CONFIG = {
    "MOCK": MOCK, "MODEL_ID": MODEL_ID, "MODEL_SOURCE": MODEL_SOURCE, "ORDER": ORDER,
    "OFFICIAL_PROMPT_FINGERPRINTS": OFFICIAL_PROMPT_FINGERPRINTS,
    "MAX_NEW_TOKENS": {"stage1": 448, "stage2": 512, "stage3": 320}, "MAX_INPUT_TOKENS": 14000,
    "VISUAL_TOKEN_BUDGET": 3200, "MIN_IMAGE_TOKENS": 64, "MAX_IMAGE_TOKENS": 1024,
    "DECODING": "greedy", "REPETITION_PENALTY": 1.05,
    "DEADLINE_EPOCH": NOTEBOOK_START + MAX_RUNTIME_HOURS * 3600,
    "MOCK_SLEEP": float(os.environ.get("TRACE_MOCK_SLEEP", "0")),
    "selection_sha256": SELECTION["sha256_questions"],
}
atomic_json(DIRS["config"] / "run_config.json", RUN_CONFIG)
atomic_json(DIRS["config"] / "selected_questions.json", SELECTION)

In [ ]:
%%writefile /kaggle/working/mammqa_trace_worker.py
# Cell 7 — worker script (one process per GPU); written to disk, launched by Cell 8
"""MAMMQA hop-trace worker: runs the released 7-agent MAMMQA pipeline on its shard of questions,
one full model copy per GPU. Launched by the notebook as:
    CUDA_VISIBLE_DEVICES=<g> python mammqa_trace_worker.py <output_root> <rank> <world>
Reads config/run_config.json and inputs/items.json (gold-free). Writes checkpoints/<qid>.json."""
import os, sys, json, re, time, math, copy, hashlib, traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple

OUT = Path(sys.argv[1]); RANK = int(sys.argv[2]); WORLD = int(sys.argv[3])
CFG = json.loads((OUT / "config" / "run_config.json").read_text(encoding="utf-8"))
ITEMS = json.loads((OUT / "inputs" / "items.json").read_text(encoding="utf-8"))
MOCK = CFG["MOCK"]
CACHE = OUT / "cache"; CKPT = OUT / "checkpoints"; LOGS = OUT / "logs"
for d in (CACHE, CKPT, LOGS):
    d.mkdir(parents=True, exist_ok=True)
LOGF = LOGS / f"worker{RANK}.log"


def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [w{RANK}] {msg}"
    print(line, flush=True)
    with open(LOGF, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def sha256_text(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def atomic_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp{RANK}")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")
    tmp.replace(path)


# ---------------------------------------------------------------- official MAMMQA prompts (prompts.py, verbatim)
Agent_stage1_system_prompt = """
You are an expert agent specialized in analyzing single-modality inputs (such as text, table, or image) and answering questions by extracting insights and systematically breaking down complex questions into simpler subquestions.

### **Task**:
You will be provided with an input and a related question. Your job is to:

Step 1:  
- Identify the modality type of the provided input.  
  Possible types: text(s), table(s), or image(s).

Step 2:  
- Clearly understand the question and carefully analyze the input.  
- Extract insights relevant to the question, example:
  - Key information (numbers, statistics, entities, trends).
  - Temporal insights (time, date, durations, timelines, etc.).

Examples for Step 2:  
- Text example insight:  
  "The text mentions that sales increased by 20 percent from January to March, highlighting quarterly growth."  
- Table example insight:  
  "The table shows the peak attendance (350 people) occurred on Saturday, June 12, 2023, indicating highest weekend engagement."  
- Image example insight:  
  "The image clearly indicates a street sign labeled '5th Avenue' and a clock showing the time as 2:15 PM, suggesting the photo was taken in mid-afternoon."

Step 3:  
- Based on these extracted insights, carefully break down the main question into simpler and more direct subquestions or counter-questions.

Examples for Step 3:  
- Main Question: "What was the monthly growth rate during Q1?"  
  - Subquestions:  
    - "What were the sales figures for January, February, and March individually?"  
    - "By how much did the sales figures change each month?"  

- Main Question: "When was attendance lowest and highest during the event period?"  
  - Subquestions:  
    - "Which date had the lowest attendance according to the provided table?"  
    - "Which date had the highest attendance according to the provided table?"

- Main Question: "At what time was the image captured?"  
  - Subquestion:  
    - "What specific time details are visible in the image?"

    
## **Important Additional Guidelines & Formatting**:

- Always think step-by-step through your analysis.
- Clearly output the identified modality type in:
  <modality> identified modality type here </modality>

- Clearly output your extracted insights in:
  <insights> your extracted insights here </insights>

- If possible, provide the final answer to original question within:
  <answer> your final answer here </answer>

- Provide answers to subquestions, wrap these in:
  <subanswer> your answer to subquestion here </subanswer>

- Only use the provided data.  
  Do not include any external or internal knowledge beyond what's explicitly given.

"""



Agent_stage2_system_prompt = """
You are an expert cross agent specialized in analyzing multiple-modalities (such as text, table, or image), insights from specialised agent(s) (such as text, table, or image) and answering questions by extracting insights and systematically breaking down complex questions into simpler subquestions.

### **Task**:
You will be provided with multiple inputs ( insights from a specialised agent(s) and multimodal input(s) ) and a related question. Your job is to:

Step 1:  
- Clearly understand the question and carefully analyze the input.  
- Extract insights relevant to the question, example:
  - Key information (numbers, statistics, entities, trends).
  - Temporal insights (time, date, durations, timelines, etc.).

Step 2:  
- Based on these extracted insights and agent insights, carefully break down the main question into simpler and more direct subquestions or counter-questions.

    
## **Important Additional Guidelines & Formatting**:

- Always think step-by-step through your analysis.

- Clearly output your extracted insights in:
  <insights> your extracted insights here </insights>

- Provide answers to subquestions, wrap these in:
  <subanswer> your answer to subquestion here </subanswer>

- Provide the final answer to original question within:
  <answer> your final answer here </answer>


- Only use the provided data.  
  Do not include any external or internal knowledge beyond what's explicitly given.

"""




Agent_stage3_system_prompt ="""
You are the final aggregator agent. Your input consists of three responses generated by cross–modal synthesis agents. Each response results from combining one modality's reasoning with the evidence from the other two modalities for the given question. Your task is to generate the most accurate final answer by following these rules:

(A) Consistency Check:

If at least two responses provide the same answer along with clear, robust reasoning, select that answer as final.

(B) Fallback Rule:

If two responses indicate that the available information is insufficient but one response gives a concrete answer with detailed evidence, choose the concrete answer.

(C) Conflict Resolution:

If all three responses differ, examine the quality of their reasoning. Weigh the clarity, depth, and coherence of the explanations, and select the answer with the strongest supporting rationale.

(D) Final Synthesis:

Provide your final answer along with a brief explanation summarizing the key points that influenced your decision.

Ensure that your decision-making is transparent, logically consistent."""

FP = {k: sha256_text(v)[:16] for k, v in {"stage1": Agent_stage1_system_prompt, "stage2": Agent_stage2_system_prompt,
                                          "stage3": Agent_stage3_system_prompt}.items()}
assert FP == CFG["OFFICIAL_PROMPT_FINGERPRINTS"], f"prompts not verbatim: {FP}"

FORMAT_ADDENDUM = (
    "\n\nFinish your response with the final answer on its own line in the form <answer>...</answer>. "
    "Inside the tags write only the short answer span (an entity, number, date, or yes/no); "
    "if the question has several answers, separate them with ' | '. No explanation inside the tags."
)

# ---------------------------------------------------------------- images (Qwen2.5-VL smart_resize)
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
PATCH = 28
MIN_T, MAX_T, BUDGET = CFG["MIN_IMAGE_TOKENS"], CFG["MAX_IMAGE_TOKENS"], CFG["VISUAL_TOKEN_BUDGET"]


def smart_resize(h, w, factor, min_pixels, max_pixels):
    if max(h, w) / max(1, min(h, w)) > 200:
        raise ValueError("aspect ratio > 200")
    hb = max(factor, round(h / factor) * factor); wb = max(factor, round(w / factor) * factor)
    if hb * wb > max_pixels:
        beta = math.sqrt((h * w) / max_pixels)
        hb = max(factor, math.floor(h / beta / factor) * factor); wb = max(factor, math.floor(w / beta / factor) * factor)
    elif hb * wb < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        hb = math.ceil(h * beta / factor) * factor; wb = math.ceil(w * beta / factor) * factor
    return hb, wb


def per_image_tokens(n):
    return 0 if n <= 0 else int(max(MIN_T, min(MAX_T, BUDGET // n)))


def load_resized(path, max_tokens):
    img = Image.open(path)
    img = img.convert("RGBA").convert("RGB") if img.mode in ("P", "LA", "RGBA") else img.convert("RGB")
    w, h = img.size
    try:
        rh, rw = smart_resize(h, w, PATCH, MIN_T * PATCH ** 2, max_tokens * PATCH ** 2)
    except ValueError:                       # extreme aspect ratio: pad to a square first
        side = max(w, h); canvas = Image.new("RGB", (side, side), (255, 255, 255)); canvas.paste(img, (0, 0)); img = canvas
        rh, rw = smart_resize(side, side, PATCH, MIN_T * PATCH ** 2, max_tokens * PATCH ** 2)
    return img.resize((rw, rh), Image.BICUBIC)


# ---------------------------------------------------------------- model
MODEL = PROCESSOR = None


def load_model():
    global MODEL, PROCESSOR
    if MOCK or MODEL is not None:
        return
    import torch
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
    src = CFG["MODEL_SOURCE"]
    log(f"loading {src} on cuda:0 (physical GPU {os.environ.get('CUDA_VISIBLE_DEVICES')})")
    PROCESSOR = AutoProcessor.from_pretrained(src, min_pixels=MIN_T * PATCH ** 2, max_pixels=MAX_T * PATCH ** 2)
    MODEL = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        src, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16),
        device_map={"": 0}, torch_dtype=torch.float16, low_cpu_mem_usage=True, attn_implementation="sdpa").eval()
    out = _generate([{"role": "user", "content": [{"type": "text", "text": "What is 2 + 3? Reply with the number only."}]}], [], 8)
    log(f"sanity check: {out['text']!r}  | GPU mem {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
    assert "5" in out["text"] or "five" in out["text"].lower(), "model sanity check failed"


def _template(messages):
    return PROCESSOR.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _count(messages, images):
    if MOCK:
        return int(len(json.dumps(messages)) / 3.6) + sum((i.size[0] // PATCH) * (i.size[1] // PATCH) for i in images)
    return int(PROCESSOR(text=[_template(messages)], images=images or None, return_tensors="pt").input_ids.shape[1])


def _generate(messages, images, max_new):
    if MOCK:
        return mock_generate(messages, max_new)
    import torch
    batch = PROCESSOR(text=[_template(messages)], images=images or None, return_tensors="pt").to(MODEL.device)
    gc = copy.deepcopy(MODEL.generation_config)
    gc.max_new_tokens = max_new; gc.repetition_penalty = CFG["REPETITION_PENALTY"]
    gc.do_sample, gc.temperature, gc.top_p, gc.top_k = False, 1.0, 1.0, 50
    gc.pad_token_id = PROCESSOR.tokenizer.pad_token_id or PROCESSOR.tokenizer.eos_token_id
    with torch.inference_mode():
        out = MODEL.generate(**batch, generation_config=gc)
    new = out[:, batch.input_ids.shape[1]:]
    text = PROCESSOR.batch_decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    n_in = int(batch.input_ids.shape[1]); del batch, out
    return {"text": text, "input_tokens": n_in, "output_tokens": int(new.shape[1])}


def _shorten(s, frac):
    keep = max(200, int(len(s) * frac))
    if keep >= len(s):
        return s
    head = int(keep * 0.7)
    return s[:head] + "\n[... evidence shortened to fit the input budget ...]\n" + s[-(keep - head):]


def call_model(spec, stage):
    """spec = {system, user:[{type:text,text,evidence?}|{type:image,path}]}; content-addressed cache = resume."""
    n_img = sum(c["type"] == "image" for c in spec["user"])
    img_tok = per_image_tokens(n_img)
    key = sha256_text(json.dumps({"model": CFG["MODEL_ID"], "max_new": CFG["MAX_NEW_TOKENS"][stage], "max_in": CFG["MAX_INPUT_TOKENS"],
                                  "rep": CFG["REPETITION_PENALTY"], "img_tok": img_tok, "spec": spec}, sort_keys=True, ensure_ascii=False))
    cpath = CACHE / f"{key}.json"
    if cpath.exists():
        return json.loads(cpath.read_text(encoding="utf-8"))
    load_model()
    user = [dict(c) for c in spec["user"]]
    images = [load_resized(c["path"], img_tok) for c in user if c["type"] == "image"]
    shortened = 0
    for _ in range(12):
        messages = [{"role": "system", "content": spec["system"]},
                    {"role": "user", "content": [{"type": "image"} if c["type"] == "image" else {"type": "text", "text": c["text"]} for c in user]}]
        if _count(messages, images) + CFG["MAX_NEW_TOKENS"][stage] <= CFG["MAX_INPUT_TOKENS"]:
            break
        ev = [i for i, c in enumerate(user) if c["type"] == "text" and c.get("evidence")]
        if not ev:
            break
        longest = max(ev, key=lambda i: len(user[i]["text"]))
        user[longest]["text"] = _shorten(user[longest]["text"], 0.8); shortened += 1
    t0 = time.perf_counter(); oom = False
    try:
        out = _generate(messages, images, CFG["MAX_NEW_TOKENS"][stage])
    except Exception as e:
        if "out of memory" not in str(e).lower():
            raise
        import torch; torch.cuda.empty_cache(); oom = True
        images = [load_resized(c["path"], max(MIN_T, img_tok // 2)) for c in user if c["type"] == "image"]
        out = _generate(messages, images, CFG["MAX_NEW_TOKENS"][stage])
    rec = {"key": key, "text": out["text"], "input_tokens": out["input_tokens"], "output_tokens": out["output_tokens"],
           "hit_max_new_tokens": out["output_tokens"] >= CFG["MAX_NEW_TOKENS"][stage], "evidence_shortenings": shortened,
           "n_images": n_img, "tokens_per_image": img_tok, "oom_fallback": oom, "seconds": round(time.perf_counter() - t0, 2)}
    atomic_json(cpath, rec)
    return rec


# ---------------------------------------------------------------- the seven agents (agents.py message formats)
def T(text, evidence=False):
    return {"type": "text", "text": text, "evidence": evidence}


def images_of(item):
    return [{"type": "image", "path": p} for p in item["image_paths"]]


def text_agent(it):
    return {"system": Agent_stage1_system_prompt, "user": [T(f"Here is the text data:\n{it['text_block']}\n Here is the question: {it['question']}", True)]}


def table_agent(it):
    return {"system": Agent_stage1_system_prompt, "user": [T(f"Here is the markdown table data:\n{it['table_block']}\n Here is the question: {it['question']}", True)]}


def image_agent(it):
    return {"system": Agent_stage1_system_prompt, "user": [T(f"Here is the question: {it['question']}")] + images_of(it)}


def text_cross_agent(it, ins):
    return {"system": Agent_stage2_system_prompt, "user": [T(f"Here is the text insight:\n{ins}\n Here is the Question: {it['question']}\n Here is the markdown table:\n{it['table_block']}", True)] + images_of(it)}


def table_cross_agent(it, ins):
    return {"system": Agent_stage2_system_prompt, "user": [T(f"Here is the table insights:\n{ins} \n\n Here is the Question: {it['question']}\n Here is the text data:\n{it['text_block']}", True)] + images_of(it)}


def image_cross_agent(it, ins):
    return {"system": Agent_stage2_system_prompt, "user": [T(f"Here is the image insights: \n {ins} \n Here is the Question: {it['question']}\n Here is the text data:\n{it['text_block']}\n Here is the markdown table: \n{it['table_block']}\n", True)]}


def reasoning_agent(it, t, tb, im):
    return {"system": Agent_stage3_system_prompt, "user": [T(f"Here is the Question: {it['question']}\n Here is the text cross output:\n{t}\n Here is the table cross output:\n{tb}\n Here is the image cross output:\n{im}" + FORMAT_ADDENDUM)]}


ANSWER_TAG = re.compile(r"<answer>(.*?)</answer>", re.I | re.S)


def final_span(raw):
    tags = [t.strip() for t in ANSWER_TAG.findall(raw or "") if t.strip()]
    if tags:
        return tags[-1], "answer_tag"
    m = re.search(r"(?is)(?:final answer|answer)\s*[:\-]\s*(.+?)(?:\n|$)", raw or "")
    if m:
        return m.group(1).strip().strip("*. "), "answer_colon"
    lines = [l.strip() for l in (raw or "").splitlines() if l.strip()]
    return (lines[-1] if lines else ""), "last_line"


def mock_generate(messages, max_new):
    blob = json.dumps(messages, ensure_ascii=False)
    h = int(sha256_text(blob)[:6], 16)
    time.sleep(CFG.get("MOCK_SLEEP", 0.0))
    words = re.findall(r"[A-Z][a-z]+(?: [A-Z][a-z]+)*", blob)
    ans = words[h % len(words)] if words else "insufficient"
    if "final aggregator" in str(messages[0]["content"]):
        cands = re.findall(r"<answer>(.*?)</answer>", blob)
        ans = cands[h % len(cands)] if cands else "unknown"
        return {"text": f"Reasoning.\n<answer>{ans}</answer>", "input_tokens": int(len(blob) / 3.6), "output_tokens": 20}
    return {"text": f"<insights> mock insight mentioning {ans} </insights>\n<answer>{ans}</answer>",
            "input_tokens": int(len(blob) / 3.6), "output_tokens": min(max_new, 40 + h % 30)}


# ---------------------------------------------------------------- run
def run_question(it):
    t0 = time.time()
    s1 = {"text": call_model(text_agent(it), "stage1"), "table": call_model(table_agent(it), "stage1"),
          "image": call_model(image_agent(it), "stage1")}
    s2 = {"text": call_model(text_cross_agent(it, s1["text"]["text"]), "stage2"),
          "table": call_model(table_cross_agent(it, s1["table"]["text"]), "stage2"),
          "image": call_model(image_cross_agent(it, s1["image"]["text"]), "stage2")}
    s3 = call_model(reasoning_agent(it, s2["text"]["text"], s2["table"]["text"], s2["image"]["text"]), "stage3")
    span, how = final_span(s3["text"])
    calls = {f"s1_{k}": v for k, v in s1.items()} | {f"s2_{k}": v for k, v in s2.items()} | {"s3": s3}
    return {
        "id": it["qid"], "question": it["question"], "prediction": span, "parse_method": how,
        "stage1": {"text": {"status": "available", "output": s1["text"]["text"]},
                   "table": {"status": "available" if it["has_table"] else "unavailable", "output": s1["table"]["text"]},
                   "image": {"status": "available" if it["image_paths"] else "unavailable", "output": s1["image"]["text"]}},
        "stage2": {k: v["text"] for k, v in s2.items()},
        "final_raw": s3["text"],
        "calls": {k: {kk: vv for kk, vv in v.items() if kk != "text"} for k, v in calls.items()},
        "n_images": len(it["image_paths"]), "missing_images": it["missing_images"],
        "worker": RANK, "seconds": round(time.time() - t0, 1),
    }


def main():
    order = CFG["ORDER"]
    mine = order[RANK::WORLD]
    deadline = CFG["DEADLINE_EPOCH"]
    log(f"{len(mine)} questions assigned; deadline in {(deadline - time.time()) / 3600:.2f} h")
    done = 0
    for qid in mine:
        ck = CKPT / f"{qid}.json"
        if ck.exists():
            continue
        if time.time() > deadline:
            log("deadline reached; not starting new questions"); break
        try:
            rec = run_question(ITEMS[qid])
            atomic_json(ck, rec); done += 1
            log(f"done {qid} in {rec['seconds']}s -> {rec['prediction'][:60]!r}")
        except Exception as e:
            atomic_json(OUT / "errors" / f"{qid}.json", {"qid": qid, "error": repr(e), "trace": traceback.format_exc()})
            log(f"ERROR {qid}: {e!r}")
            if not MOCK:
                import torch; torch.cuda.empty_cache()
    log(f"finished: {done} new questions")


if __name__ == "__main__":
    main()

In [ ]:
# Cell 8 — launch one worker per GPU and monitor until all finish
WORKER = Path(os.environ.get("TRACE_WORKER_PATH", "/kaggle/working/mammqa_trace_worker.py"))   # written by Cell 7
assert WORKER.exists(), "run Cell 7 first"
def launch(rank, world):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(rank), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    out = open(DIRS["logs"] / f"worker{rank}.stdout", "a")
    return subprocess.Popen([sys.executable, str(WORKER), str(OUTPUT_ROOT), str(rank), str(world)], env=env, stdout=out, stderr=subprocess.STDOUT)

WORLD = max(1, N_GPU)
procs = {r: launch(r, WORLD) for r in range(WORLD)}
restarts = Counter()
last_report, t_start = 0.0, time.time()
done0 = len(list(DIRS["checkpoints"].glob("*.json")))
while procs:
    time.sleep(2 if MOCK else 60)
    for r, p in list(procs.items()):
        code = p.poll()
        if code is None:
            continue
        del procs[r]
        mine_left = [q for q in ORDER[r::WORLD] if not (DIRS["checkpoints"] / f"{q}.json").exists()]
        if code != 0 and mine_left and restarts[r] < 2 and time.time() < RUN_CONFIG["DEADLINE_EPOCH"]:
            restarts[r] += 1
            log(f"worker {r} exited with code {code}; restarting ({restarts[r]}/2)")
            procs[r] = launch(r, WORLD)
        else:
            log(f"worker {r} finished (exit {code}); {len(mine_left)} of its questions not done")
    done = len(list(DIRS["checkpoints"].glob("*.json")))
    if time.time() - last_report > (5 if MOCK else 600) or not procs:
        rate = (done - done0) / max(1e-9, (time.time() - t_start) / 3600)
        eta = (len(ORDER) - done) / rate if rate > 0 else float("nan")
        log(f"progress {done}/{len(ORDER)} | {rate:.1f} q/h | ETA {eta:.1f} h | errors {len(list(DIRS['errors'].glob('*.json')))}")
        last_report = time.time()

In [ ]:
# Cell 9 — FREEZE predictions (hash) before any gold is read
done_qids = [q for q in ORDER if (DIRS["checkpoints"] / f"{q}.json").exists()]
PRED_PATH = DIRS["predictions"] / "mammqa_predictions.jsonl"
with open(PRED_PATH, "w", encoding="utf-8") as f:
    for q in done_qids:
        f.write(json.dumps(json.loads((DIRS["checkpoints"] / f"{q}.json").read_text(encoding="utf-8")), ensure_ascii=False, sort_keys=True) + "\n")
FROZEN = {"n": len(done_qids), "of": len(ORDER), "sha256": hashlib.sha256(PRED_PATH.read_bytes()).hexdigest(),
          "selection_sha256": SELECTION["sha256_questions"], "frozen_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
atomic_json(DIRS["predictions"] / "PREDICTIONS_FROZEN.json", FROZEN)
log(f"PREDICTIONS FROZEN: {FROZEN['n']}/{FROZEN['of']} sha256 {FROZEN['sha256'][:16]} — gold evaluation starts now")

In [ ]:
# Cell 10 — official MMQA EM/F1 (DROP-style) after the freeze; the hop audit runs on mammqa_predictions.jsonl
def _is_num(t):
    try: float(t); return True
    except ValueError: return False
_P = set(string.punctuation)
def _nt(t):
    t = t.lower()
    if not _is_num(t): t = "".join(c for c in t if c not in _P)
    t = re.sub(r"\b(a|an|the)\b", " ", t)
    if _is_num(t): t = str(float(t))
    return " ".join(t.split())
def _norm(s): return " ".join(x for x in (_nt(tok) for tok in re.split(" |-", str(s))) if x.strip()).strip()
def _f1(p, g):
    i = len(g & p); pr = 1.0 if not p else i / len(p); rc = 1.0 if not g else i / len(g)
    return 0.0 if pr == 0 and rc == 0 else 2 * pr * rc / (pr + rc)
def em_f1(pred, gold):
    from itertools import permutations
    pn = [_norm(x) for x in pred]; gn = [_norm(x) for x in gold]
    em = float(set(pn) == set(gn) and len(pn) == len(gn))
    pb = [set(x.split()) for x in pn]; gb = [set(x.split()) for x in gn]
    def ok(g, p):
        gnum = {w for w in g if _is_num(w)}
        return (not gnum) or bool(gnum & {w for w in p if _is_num(w)})
    S = [[_f1(p, g) if ok(g, p) else 0.0 for p in pb] for g in gb]
    try:
        from scipy.optimize import linear_sum_assignment; import numpy as np
        r, c = linear_sum_assignment(-np.array(S)); pairs = list(zip(r, c))
    except Exception:
        pairs = [(i, max(range(len(pb)), key=lambda j: S[i][j])) for i in range(len(gb))]
    best = [0.0] * max(len(gb), len(pb))
    for i, j in pairs: best[i] = max(best[i], S[i][j])
    return em, sum(best) / len(best)
def fmt(v): return str(int(v)) if isinstance(v, float) and v.is_integer() else str(v)

gold = {r["qid"]: r for r in read_jsonl(MMQA["dev"]) if r["qid"] in set(done_qids)}
rows = []
for line in open(PRED_PATH, encoding="utf-8"):
    p = json.loads(line); g = gold[p["id"]]
    golds = [fmt(a["answer"]) for a in g["answers"]]
    parts = [x.strip() for x in re.split(r"\s*\|\s*|\s*;\s*", p["prediction"]) if x.strip()] or [p["prediction"]]
    em, f1 = em_f1(parts, golds)
    rows.append({"qid": p["id"], "type": g["metadata"]["type"], "top_type": g["metadata"]["type"].split("(")[0], "em": em, "f1": round(f1, 4),
                 "prediction": p["prediction"], "gold": " | ".join(golds), "seconds": p["seconds"], "n_images": p["n_images"],
                 "hit_cap": sum(c["hit_max_new_tokens"] for c in p["calls"].values()),
                 "shortened": sum(c["evidence_shortenings"] for c in p["calls"].values())})
import csv
if rows:
    with open(DIRS["metrics"] / "per_question.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
by_type = defaultdict(list)
for r in rows: by_type[r["top_type"]].append(r["em"])
n_rows = max(1, len(rows))
SUMMARY = {"n": len(rows), "EM": round(sum(r["em"] for r in rows) / n_rows, 4), "F1": round(sum(r["f1"] for r in rows) / n_rows, 4),
           "EM_by_type": {t: [round(sum(v) / len(v), 4), len(v)] for t, v in sorted(by_type.items())},
           "mean_seconds_per_question": round(sum(r["seconds"] for r in rows) / n_rows, 1),
           "calls_hitting_token_cap": sum(r["hit_cap"] for r in rows), "evidence_shortenings": sum(r["shortened"] for r in rows),
           "errors": [p.stem for p in DIRS["errors"].glob("*.json") if not (DIRS["checkpoints"] / p.name).exists()],
           "frozen": FROZEN, "wall_hours": round((time.time() - NOTEBOOK_START) / 3600, 2)}
atomic_json(DIRS["metrics"] / "summary.json", SUMMARY)
print(json.dumps(SUMMARY, indent=1))